## 1. Imports et configuration

In [ ]:
import os, re, json, math, random, hashlib, unicodedata, platform, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)

STEP2_DIR = Path("outputs/step2_index")          # produit par le notebook de l'étape 2
SEED_PATH = Path("train_seed_b1.jsonl")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step3_dataset"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_FRACTION = 0.20            # part des questions-sources réservée à la validation (split PAR QUESTION, jamais par exemple)
N_VARIANTS_TRAIN = 2           # nombre de contextes différents générés par question d'entraînement
MISS_RATE = 0.15               # part des questions « avec réponse » dont on génère AUSSI une variante « retrieval manqué » (réponse absente du contexte -> abstention)
MISS_ANSWER_OVERLAP = 0.5       # une variante « miss » est écartée si ≥ 50 % du contenu de la réponse est encore dans un passage du contexte (le guide répète certains faits)
ABSTAIN_MARKER = "INFORMATION ABSENTE"          # identique à l'étape 1 (les métriques d'abstention en dépendent)
ABSTAIN_ANSWER = ABSTAIN_MARKER + " : les passages fournis ne contiennent pas cette information."
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

random.seed(SEED); np.random.seed(SEED)
rng = random.Random(SEED)
print("python", platform.python_version())

## 2. Utilitaires (identiques à l'étape 2 : normalisation, BM25, citation)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

STOPWORDS = set("le la les l un une des de du d et ou a au aux en dans par pour sur sous avec sans que qui quoi est sont etre ce cet cette ces se sa son ses "
                "leur leurs ne pas plus il ils elle elles on y c s n qu j m t si comme mais donc car faut faire peut doit quel quelle quels quelles".split())
def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", norm(text))
    return [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in toks if t not in STOPWORDS]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(d) for d in docs_tokens]
        self.len = np.array([len(d) for d in docs_tokens], dtype=float); self.avg = self.len.mean()
        df = Counter(t for d in self.docs for t in d); n = len(docs_tokens)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def scores(self, q_tokens):
        s = np.zeros(len(self.docs))
        for t in set(q_tokens):
            if t in self.idf:
                tf = np.array([d.get(t, 0) for d in self.docs], dtype=float)
                s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

def cite(c):
    p = [x for x in c["pages"] if x >= 1]            # la page de présentation du PDF n'a pas de numéro imprimé
    pages = "page non numérotée" if not p else (f"p. {p[0]}" if len(p) == 1 else f"p. {p[0]}-{p[-1]}")
    where = "Présentation du guide" if c["fiche"] == 0 else f"Fiche {c['fiche']} — {c['fiche_title']}"
    return f"{where} — {c['section_label']}, {pages}"

from functools import lru_cache
@lru_cache(maxsize=None)
def rx(pattern):
    """Regex compilée sur le texte normalisé (mise en cache : le scan de fuite l'appelle des centaines de milliers de fois)."""
    return re.compile(norm(pattern))

def jaccard(a, b):
    a, b = set(tokenize(a)), set(tokenize(b))
    return len(a & b) / max(len(a | b), 1)

## 3. Chargement de l'index (étape 2) et des jeux de données


In [ ]:
index_cfg = json.load(open(STEP2_DIR / "index_config.json", encoding="utf-8"))
chunks = load_jsonl(STEP2_DIR / "chunks.jsonl")
K_CONTEXT = int(index_cfg["k_final"])
assert K_CONTEXT >= 3, f"k_final={K_CONTEXT} : trop petit pour une réponse réglementaire complète — relancer l'étape 2 (règle de complétude rep@k)."
print(f"{len(chunks)} chunks | config étape 2 : chunking={index_cfg['chunking']} embedder={index_cfg['embedder']} méthode={index_cfg['method']} | k={K_CONTEXT}")
chunk_norm = [norm(c["text"]) for c in chunks]
bm25 = BM25([tokenize(c["embed_text"]) for c in chunks])

seed_rows = load_jsonl(SEED_PATH); test_rows = load_jsonl(TEST_SET_PATH); dev_rows = load_jsonl(DEV_SET_PATH)
SEED_SHA = hashlib.sha256(SEED_PATH.read_bytes()).hexdigest(); TEST_SHA = hashlib.sha256(TEST_SET_PATH.read_bytes()).hexdigest()
if index_cfg.get("test_sha256") and index_cfg["test_sha256"] != TEST_SHA:
    print("[!] ATTENTION : le test_set_b1.jsonl actuel n'est pas celui utilisé à l'étape 2 (hash différent). Vérifier qu'il s'agit bien du même fichier.")
print(f"graine : {len(seed_rows)} questions | test : {len(test_rows)} | dev : {len(dev_rows)}")
print(Counter(r["type"] for r in seed_rows))

## 4. Garde-fous anti-fuite

### 4.1 Questions : aucune question commune ou quasi-identique avec le test ou le dev

In [ ]:
held_out = [("test", r["question"]) for r in test_rows] + [("dev", r["question"]) for r in dev_rows]
violations = []
for r in seed_rows:
    for src, q in held_out:
        if norm(r["question"]) == norm(q) or jaccard(r["question"], q) >= 0.8:
            violations.append((r["id"], src, r["question"], q))
print("Questions d'entraînement identiques/quasi-identiques à test/dev :", violations or "aucune")
assert not violations

### 4.2 Passages : les passages qui contiennent la réponse d'une question de test sont **interdits** dans tout contexte d'entraînement

In [ ]:
def group_rate(alts, chunk_norm):
    return np.mean([any(rx(a).search(t) for a in alts) for t in chunk_norm])

def informative_groups(key_facts, chunk_norm, max_rate=0.15):
    return [alts for alts in key_facts if group_rate(alts, chunk_norm) <= max_rate]

def blocked_chunk_ids(test_rows, chunks, chunk_norm):
    blocked, why = set(), defaultdict(list)
    for r in test_rows:
        if r["category"] != "in_corpus":
            continue
        info = informative_groups(r["key_facts"], chunk_norm)
        specific_single = len(info) == 1 and group_rate(info[0], chunk_norm) <= 0.05
        for j, c in enumerate(chunks):
            rule1 = (c["fiche"] in r["gold_fiches"] and set(c["pages"]) & set(r["gold_pages"])
                     and any(rx(a).search(chunk_norm[j]) for alts in r["key_facts"] for a in alts))
            rule2 = bool(info) and (len(info) >= 2 or specific_single) and \
                    all(any(rx(a).search(chunk_norm[j]) for a in alts) for alts in info)
            if rule1 or rule2:
                blocked.add(c["chunk_id"]); why[c["chunk_id"]].append((r["id"], "zone" if rule1 else "doublon"))
    return blocked, why

BLOCKED, BLOCK_WHY = blocked_chunk_ids(test_rows, chunks, chunk_norm)
usable = [j for j, c in enumerate(chunks) if c["chunk_id"] not in BLOCKED]
n_dup = sum(1 for cid, w in BLOCK_WHY.items() if all(k == "doublon" for _, k in w))
print(f"{len(BLOCKED)} chunks bloqués sur {len(chunks)} ({len(BLOCKED)/len(chunks):.0%}) | dont {n_dup} uniquement par la règle « fait dupliqué ailleurs » | {len(usable)} utilisables pour l'entraînement")
cov = pd.DataFrame({"fiche": [c["fiche"] for c in chunks], "bloqué": [c["chunk_id"] in BLOCKED for c in chunks]})
display(cov.groupby("fiche").bloqué.agg(["sum", "count"]).rename(columns={"sum": "bloqués", "count": "chunks"}).T)

## 5. Format du prompt RAG (source de vérité pour les étapes 3, 4, 5 et 6)

In [ ]:
SYSTEM_PROMPT = (
    "Tu es un assistant réglementaire spécialisé dans la sécurité des données personnelles. Tu réponds en français. "
    "Tu réponds UNIQUEMENT à partir des passages fournis, extraits du « Guide pratique RGPD – Sécurité des données personnelles » de la CNIL. "
    "N'utilise aucune connaissance extérieure aux passages.\n"
    "Règles :\n"
    f"1. Si les passages ne contiennent pas l'information demandée, réponds exactement : {ABSTAIN_ANSWER}\n"
    "2. Sinon, réponds en trois phrases maximum, sans rien ajouter qui ne figure pas dans les passages, "
    "puis termine par une dernière ligne « Sources : [n] » listant uniquement les numéros des passages utilisés (ex. : Sources : [2], [4])."
)
USER_TEMPLATE = "Passages :\n\n{passages}\n\nQuestion : {question}"
PASSAGE_TEMPLATE = "[{n}] ({citation})\n{text}"

def build_messages(question, passages, answer=None):
    """passages : liste de chunks (dict). Retourne la liste de messages chat (avec la réponse si fournie)."""
    ctx = "\n\n".join(PASSAGE_TEMPLATE.format(n=i + 1, citation=cite(c), text=c["text"]) for i, c in enumerate(passages))
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(passages=ctx, question=question)}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    return msgs

json.dump({"system": SYSTEM_PROMPT, "user_template": USER_TEMPLATE, "passage_template": PASSAGE_TEMPLATE,
           "abstain_marker": ABSTAIN_MARKER, "abstain_answer": ABSTAIN_ANSWER, "k_context": K_CONTEXT},
          open(OUTPUT_DIR / "rag_prompt.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(build_messages("Exemple de question ?", chunks[:2])[1]["content"][:700])